In [ ]:
import os
import copy
import json
import random
import warnings
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset

warnings.filterwarnings("ignore")
os.makedirs("./outputs", exist_ok=True)

# =========================
# CONFIG
# =========================
QUICK_RUN = True
SAVE_PATH = "./outputs/lwf_progress_stable.json"

if QUICK_RUN:
    EPOCHS_A = 25
    EPOCHS_B = 45
    EPOCHS_BIAS = 5
    SEEDS = [0]
else:
    EPOCHS_A = 60
    EPOCHS_B = 120
    EPOCHS_BIAS = 10
    SEEDS = [0, 1, 2, 3, 4]

BS = 128
NW = 2
LR_A = 1e-3
LR_B = 8e-4
LR_BIAS = 5e-5
WEIGHT_DECAY = 1e-4

# LwF params
KD_T = 2.0
KD_LAMBDA = 1.2   # modest distillation
NEW_CE_W = 1.6    # prioritize new class learning a bit to avoid B collapse

# Small balanced memory for calibration only (not replay training)
CAL_PER_CLASS_OLD = 120
CAL_PER_CLASS_NEW = 120

# =========================
# UTILS
# =========================
def set_deterministic(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

def get_class_subset(dataset, classes):
    idx = [i for i, y in enumerate(dataset.targets) if y in classes]
    return Subset(dataset, idx)

def build_class_index_map(dataset):
    m = {}
    for i, y in enumerate(dataset.targets):
        m.setdefault(y, []).append(i)
    return m

def build_balanced_subset(dataset, class_to_indices, classes, per_class, seed=0):
    rng = random.Random(seed)
    idx = []
    for c in classes:
        pool = class_to_indices[c]
        if len(pool) <= per_class:
            idx.extend(pool)
        else:
            idx.extend(rng.sample(pool, per_class))
    rng.shuffle(idx)
    return Subset(dataset, idx)

def save_progress(obj, path=SAVE_PATH):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

# =========================
# MODEL (simple ResNet18 backbone)
# =========================
def make_resnet18(num_classes):
    m = torchvision.models.resnet18(weights=None)
    m.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    m.maxpool = nn.Identity()
    m.fc = nn.Linear(512, num_classes)
    return m

# =========================
# LOSSES
# =========================
def kd_loss(student_old_logits, teacher_old_logits, T=2.0):
    # KL(student || teacher) on old classes only
    p_s = F.log_softmax(student_old_logits / T, dim=1)
    p_t = F.softmax(teacher_old_logits / T, dim=1)
    return F.kl_div(p_s, p_t, reduction="batchmean") * (T * T)

# =========================
# EVAL
# =========================
def evaluate(model, loader, device, mode="cil"):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)

            if mode == "task_A":
                pred = logits[:, :50].argmax(1)
            elif mode == "task_B":
                pred = logits[:, 50:].argmax(1) + 50
            else:
                pred = logits.argmax(1)

            correct += (pred == y).sum().item()
            total += y.size(0)
    return 100.0 * correct / total

# =========================
# MAIN EXPERIMENT
# =========================
def run_lwf(seed, loaders, cal_loader, device):
    set_deterministic(seed)
    train_A, train_B, test_A, test_B = loaders

    # ----- Phase 1: Train task A model (50-way)
    model_A = make_resnet18(num_classes=50).to(device)
    optA = torch.optim.AdamW(model_A.parameters(), lr=LR_A, weight_decay=WEIGHT_DECAY)
    schA = torch.optim.lr_scheduler.CosineAnnealingLR(optA, T_max=EPOCHS_A)

    for _ in range(EPOCHS_A):
        model_A.train()
        for x, y in train_A:
            x, y = x.to(device), y.to(device)
            optA.zero_grad()
            logits = model_A(x)
            loss = F.cross_entropy(logits, y, label_smoothing=0.1)
            loss.backward()
            optA.step()
        schA.step()

    acc_A_init = evaluate(model_A, test_A, device, mode="task_A")

    # Freeze teacher
    teacher = copy.deepcopy(model_A).to(device).eval()
    for p in teacher.parameters():
        p.requires_grad = False

    # ----- Build student 100-way, copy shared weights
    student = make_resnet18(num_classes=100).to(device)
    sd = student.state_dict()
    td = teacher.state_dict()

    # copy everything except fc
    for k in sd.keys():
        if not k.startswith("fc."):
            sd[k] = td[k]
    # copy old classifier weights
    with torch.no_grad():
        sd["fc.weight"][:50] = td["fc.weight"]
        sd["fc.bias"][:50] = td["fc.bias"]
        nn.init.kaiming_normal_(sd["fc.weight"][50:])
        sd["fc.bias"][50:].zero_()
    student.load_state_dict(sd)

    # ----- Phase 2: LwF on task B
    optB = torch.optim.AdamW(student.parameters(), lr=LR_B, weight_decay=WEIGHT_DECAY)
    schB = torch.optim.lr_scheduler.CosineAnnealingLR(optB, T_max=EPOCHS_B)

    for _ in range(EPOCHS_B):
        student.train()
        for x, y in train_B:
            x, y = x.to(device), y.to(device)
            y_new = y - 50  # 0..49 for new slice
            optB.zero_grad()

            logits_s = student(x)                # 100-way
            with torch.no_grad():
                logits_t_old = teacher(x)        # 50-way

            loss_new = F.cross_entropy(logits_s[:, 50:], y_new, label_smoothing=0.05)
            loss_kd  = kd_loss(logits_s[:, :50], logits_t_old, T=KD_T)

            loss = NEW_CE_W * loss_new + KD_LAMBDA * loss_kd
            loss.backward()
            optB.step()
        schB.step()

    # ----- Phase 3: Tiny calibration on balanced old+new memory (classifier only)
    for p in student.parameters():
        p.requires_grad = False
    student.fc.weight.requires_grad = True
    student.fc.bias.requires_grad = True

    optC = torch.optim.AdamW([student.fc.weight, student.fc.bias], lr=LR_BIAS, weight_decay=1e-5)
    schC = torch.optim.lr_scheduler.CosineAnnealingLR(optC, T_max=EPOCHS_BIAS)

    for _ in range(EPOCHS_BIAS):
        student.train()
        for x, y in cal_loader:
            x, y = x.to(device), y.to(device)
            optC.zero_grad()
            logits = student(x)
            loss = F.cross_entropy(logits, y, label_smoothing=0.0)
            loss.backward()
            optC.step()
        schC.step()

    # ----- Eval
    acc_A_final_ta = evaluate(student, test_A, device, mode="task_A")
    acc_B_final_ta = evaluate(student, test_B, device, mode="task_B")
    acc_A_final_cil = evaluate(student, test_A, device, mode="cil")
    acc_B_final_cil = evaluate(student, test_B, device, mode="cil")

    return {
        "acc_A_init": acc_A_init,
        "acc_A_final_taskaware": acc_A_final_ta,
        "acc_B_final_taskaware": acc_B_final_ta,
        "acc_A_final_cil": acc_A_final_cil,
        "acc_B_final_cil": acc_B_final_cil,
        "retention_taskaware": (acc_A_final_ta / acc_A_init) * 100 if acc_A_init > 0 else 0.0,
        "retention_cil": (acc_A_final_cil / acc_A_init) * 100 if acc_A_init > 0 else 0.0,
        "forgetting_taskaware": acc_A_init - acc_A_final_ta,
        "forgetting_cil": acc_A_init - acc_A_final_cil,
        "bwt_taskaware": acc_A_final_ta - acc_A_init,
        "bwt_cil": acc_A_final_cil - acc_A_init
    }

def run_all():
    device = get_device()
    print(f"🚀 Environment: {device} | QUICK_RUN={QUICK_RUN}")

    stats = ((0.5071, 0.4865, 0.4409), (0.2673, 0.2564, 0.2762))
    t_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(*stats),
    ])
    t_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(*stats),
    ])

    root = "/kaggle/input/cifar-100" if os.path.exists("/kaggle/input/cifar-100") else "./data"
    train_ds = torchvision.datasets.CIFAR100(root=root, train=True, download=True, transform=t_train)
    test_ds = torchvision.datasets.CIFAR100(root=root, train=False, download=True, transform=t_test)

    train_A = DataLoader(get_class_subset(train_ds, range(50)), batch_size=BS, shuffle=True, num_workers=NW, pin_memory=True, persistent_workers=True)
    train_B = DataLoader(get_class_subset(train_ds, range(50, 100)), batch_size=BS, shuffle=True, num_workers=NW, pin_memory=True, persistent_workers=True)
    test_A  = DataLoader(get_class_subset(test_ds, range(50)), batch_size=BS, shuffle=False, num_workers=NW, pin_memory=True)
    test_B  = DataLoader(get_class_subset(test_ds, range(50, 100)), batch_size=BS, shuffle=False, num_workers=NW, pin_memory=True)

    # balanced calibration loader old+new
    class_to_idx = build_class_index_map(train_ds)
    cal_old = build_balanced_subset(train_ds, class_to_idx, range(50), CAL_PER_CLASS_OLD, seed=123)
    cal_new = build_balanced_subset(train_ds, class_to_idx, range(50, 100), CAL_PER_CLASS_NEW, seed=456)

    # merge subsets
    merged_idx = []
    merged_idx.extend(cal_old.indices if isinstance(cal_old, Subset) else [])
    merged_idx.extend(cal_new.indices if isinstance(cal_new, Subset) else [])
    cal_subset = Subset(train_ds, merged_idx)
    cal_loader = DataLoader(cal_subset, batch_size=BS, shuffle=True, num_workers=NW, pin_memory=True)

    loaders = (train_A, train_B, test_A, test_B)

    results = []
    meta = {
        "timestamp": str(datetime.utcnow()),
        "quick_run": QUICK_RUN,
        "epochs_A": EPOCHS_A,
        "epochs_B": EPOCHS_B,
        "epochs_bias": EPOCHS_BIAS,
        "seeds": SEEDS,
        "results": []
    }

    print("\n[1] Running LwF experiments...")
    try:
        for s in SEEDS:
            print(f" -> Seed {s}...")
            r = run_lwf(s, loaders, cal_loader, device)
            results.append(r)
            meta["results"].append({"seed": s, **r})
            save_progress(meta)

            print(
                f"    ↳ Seed {s}: "
                f"A_init(TA)={r['acc_A_init']:.2f} | "
                f"A_final(TA)={r['acc_A_final_taskaware']:.2f} | "
                f"B_final(TA)={r['acc_B_final_taskaware']:.2f} | "
                f"A_final(CIL)={r['acc_A_final_cil']:.2f} | "
                f"B_final(CIL)={r['acc_B_final_cil']:.2f}"
            )

    except KeyboardInterrupt:
        print("\n⛔ Interrupted. Partial results saved to:", SAVE_PATH)
        save_progress(meta)
        return

    if results:
        print("\n" + "=" * 90)
        print("SUMMARY (Mean ± Std)")
        print("=" * 90)

        for k, label in [
            ("acc_A_init", "A Init (Task-aware)"),
            ("acc_A_final_taskaware", "A Final (Task-aware)"),
            ("acc_B_final_taskaware", "B Final (Task-aware)"),
            ("acc_A_final_cil", "A Final (CIL)"),
            ("acc_B_final_cil", "B Final (CIL)")
        ]:
            vals = [r[k] for r in results]
            print(f"{label:26}: {np.mean(vals):.2f} ± {np.std(vals):.2f}")

        for k, label in [
            ("retention_taskaware", "Retention (Task-aware)"),
            ("retention_cil", "Retention (CIL)"),
            ("forgetting_taskaware", "Forgetting (Task-aware)"),
            ("forgetting_cil", "Forgetting (CIL)"),
            ("bwt_taskaware", "BWT (Task-aware)"),
            ("bwt_cil", "BWT (CIL)")
        ]:
            vals = [r[k] for r in results]
            print(f"{label:26}: {np.mean(vals):.2f} ± {np.std(vals):.2f}")

        print("=" * 90)
        print(f"Saved run log to: {SAVE_PATH}")

if __name__ == "__main__":
    run_all()

In [ ]:
'''Results 
ubuntu@gt-ubuntu24-04-cmd-v3-2-120gb-100m:~$ python3 Replay.py
🚀 Environment: cuda | QUICK_RUN=True

[1] Running LwF experiments...
 -> Seed 0...
    ↳ Seed 0: A_init(TA)=75.50 | A_final(TA)=76.02 | B_final(TA)=76.46 | A_final(CIL)=67.68 | B_final(CIL)=62.80

==========================================================================================
SUMMARY (Mean ± Std)
==========================================================================================
A Init (Task-aware)       : 75.50 ± 0.00
A Final (Task-aware)      : 76.02 ± 0.00
B Final (Task-aware)      : 76.46 ± 0.00
A Final (CIL)             : 67.68 ± 0.00
B Final (CIL)             : 62.80 ± 0.00
Retention (Task-aware)    : 100.69 ± 0.00
Retention (CIL)           : 89.64 ± 0.00
Forgetting (Task-aware)   : -0.52 ± 0.00
Forgetting (CIL)          : 7.82 ± 0.00
BWT (Task-aware)          : 0.52 ± 0.00
BWT (CIL)                 : -7.82 ± 0.00
==========================================================================================
Saved run log to: ./outputs/lwf_progress_stable.json

'''